# DeepShield v2 — Treino Multi-Gerador no Google Colab

**Objetivo:** Re-treinar o DeepShield com datasets de multiplos geradores de IA (StyleGAN, DALL-E, Stable Diffusion, Midjourney, ProGAN) para melhorar a generalizacao.

**Datasets:**
1. `xhlulu/140k-real-and-fake-faces` — StyleGAN (100k imagens)
2. `awsaf49/artifact-dataset` — ArtiFact (DALL-E, Stable Diffusion, Midjourney, ProGAN, StyleGAN)

**Estrategia:** Fase 1 (backbone congelado) + Fase 2 (fine-tune ultimos 3 blocos)

## 1. Verificar GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponivel: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    raise RuntimeError('GPU nao encontrada! Va em Runtime > Change runtime type > GPU')

## 2. Clonar repositorio e instalar dependencias

In [ ]:
!git clone https://github.com/GuizanzotiSB/deepshield.git
%cd deepshield
!pip install -q -r requirements.txt

## 3. Configurar Kaggle API

Preencha seu username e key abaixo. **NAO commite este notebook com a key preenchida.**

In [ ]:
import os

os.makedirs("/root/.kaggle", exist_ok=True)
kaggle_user = input("Kaggle username: ")
kaggle_key = input("Kaggle API key: ")
with open("/root/.kaggle/kaggle.json", "w") as f:
    f.write(f'{{"username":"{kaggle_user}","key":"{kaggle_key}"}}')
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print(f"Kaggle configurado para: {kaggle_user}")

## 4. Baixar datasets

In [ ]:
!pip install -q kaggle

# Dataset 1: 140k Real and Fake Faces (StyleGAN)
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p data/raw/stylegan --unzip

# Dataset 2: ArtiFact (multi-gerador)
!kaggle datasets download -d awsaf49/artifact-dataset -p data/raw/artifact --unzip

## 5. Explorar estrutura dos datasets

In [ ]:
import os
from pathlib import Path
from collections import Counter

print('='*60)
print('DATASET 1: 140k Real and Fake Faces (StyleGAN)')
print('='*60)
stylegan_root = Path('data/raw/stylegan')
for d in sorted(stylegan_root.rglob('*')):
    if d.is_dir():
        n_files = len(list(d.glob('*')))
        if n_files > 0:
            print(f'  {d.relative_to(stylegan_root)}: {n_files} arquivos')

print()
print('='*60)
print('DATASET 2: ArtiFact (multi-gerador)')
print('='*60)
artifact_root = Path('data/raw/artifact')
for d in sorted(artifact_root.rglob('*')):
    if d.is_dir():
        n_files = len([f for f in d.glob('*') if f.is_file()])
        if n_files > 0:
            print(f'  {d.relative_to(artifact_root)}: {n_files} arquivos')

In [ ]:
# Explorar subpastas do ArtiFact em mais detalhe
print('Estrutura de pastas (ate nivel 3):')
for d in sorted(artifact_root.rglob('*')):
    if d.is_dir():
        depth = len(d.relative_to(artifact_root).parts)
        if depth <= 3:
            indent = '  ' * depth
            n_img = len([f for f in d.iterdir() if f.suffix.lower() in ('.jpg','.jpeg','.png','.webp')])
            n_sub = len([f for f in d.iterdir() if f.is_dir()])
            info = f'{n_img} imgs' if n_img > 0 else f'{n_sub} subdirs'
            print(f'{indent}{d.name}/ ({info})')

## 6. Criar dataset unificado multi-gerador

Combina imagens de ambos datasets em uma estrutura unica `data/unified/{train,val}/{real,fake}/`.

In [ ]:
import shutil
import random
from pathlib import Path
from collections import defaultdict

random.seed(42)

UNIFIED_DIR = Path('data/unified')
UNIFIED_DIR.mkdir(parents=True, exist_ok=True)
for split in ('train', 'val'):
    for cls in ('real', 'fake'):
        (UNIFIED_DIR / split / cls).mkdir(parents=True, exist_ok=True)

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tiff'}
stats = defaultdict(lambda: defaultdict(int))  # stats[source][class] = count

def collect_images(directory, label, source_name, max_per_source=None):
    """Coleta imagens de um diretorio e retorna lista de (path, label, source)."""
    imgs = [f for f in Path(directory).rglob('*') if f.suffix.lower() in IMG_EXTS]
    if max_per_source and len(imgs) > max_per_source:
        imgs = random.sample(imgs, max_per_source)
    return [(img, label, source_name) for img in imgs]

# --- Dataset 1: 140k Real and Fake Faces ---
all_images = []

stylegan_base = Path('data/raw/stylegan/real_vs_fake/real-vs-fake')
# Tenta caminhos alternativos caso a estrutura seja diferente
if not stylegan_base.exists():
    stylegan_base = Path('data/raw/stylegan')
    # Procura subpastas 'real' e 'fake'
    for candidate in stylegan_base.rglob('real'):
        if candidate.is_dir():
            stylegan_base = candidate.parent
            break

for split_name in ('train', 'test', 'valid'):
    split_dir = stylegan_base / split_name
    if not split_dir.exists():
        continue
    real_dir = split_dir / 'real'
    fake_dir = split_dir / 'fake'
    if real_dir.exists():
        all_images.extend(collect_images(real_dir, 'real', f'stylegan_{split_name}'))
    if fake_dir.exists():
        all_images.extend(collect_images(fake_dir, 'fake', f'stylegan_{split_name}'))

print(f'StyleGAN: {len(all_images)} imagens coletadas')

# --- Dataset 2: ArtiFact ---
artifact_base = Path('data/raw/artifact')
artifact_count_before = len(all_images)

# ArtiFact tem estrutura variada — busca recursiva por pastas 'real'/'fake'
# e tambem por nomes de geradores
FAKE_KEYWORDS = {'fake', 'generated', 'synthetic', 'gan', 'diffusion',
                 'dalle', 'dall-e', 'midjourney', 'stable_diffusion',
                 'stylegan', 'progan', 'biggan', 'stargan', 'glide',
                 'sdv', 'ldm', 'deepfake', 'manipulated'}
REAL_KEYWORDS = {'real', 'authentic', 'original', 'nature', 'celeba',
                 'ffhq', 'lsun', 'imagenet', 'coco'}

for folder in sorted(artifact_base.rglob('*')):
    if not folder.is_dir():
        continue
    folder_lower = folder.name.lower()
    imgs = [f for f in folder.iterdir() if f.suffix.lower() in IMG_EXTS]
    if len(imgs) == 0:
        continue

    # Determina label baseado no nome da pasta
    label = None
    source = f'artifact_{folder.name}'

    # Checa caminho completo (ex: artifact/train/fake/stylegan)
    full_path_lower = str(folder.relative_to(artifact_base)).lower()
    if any(kw in full_path_lower for kw in FAKE_KEYWORDS):
        label = 'fake'
    elif any(kw in full_path_lower for kw in REAL_KEYWORDS):
        label = 'real'

    if label:
        # Limita a 10k por sub-fonte para balanceamento
        all_images.extend(collect_images(folder, label, source, max_per_source=10000))

artifact_count = len(all_images) - artifact_count_before
print(f'ArtiFact: {artifact_count} imagens coletadas')
print(f'Total: {len(all_images)} imagens')

In [ ]:
# Estatisticas por fonte e classe
from collections import Counter
import pandas as pd

source_label_counts = Counter((source, label) for _, label, source in all_images)
source_stats = defaultdict(lambda: {'real': 0, 'fake': 0})
for (source, label), count in source_label_counts.items():
    source_stats[source][label] = count

df_stats = pd.DataFrame([
    {'Fonte': src, 'Real': counts['real'], 'Fake': counts['fake'],
     'Total': counts['real'] + counts['fake']}
    for src, counts in sorted(source_stats.items())
])
print(df_stats.to_string(index=False))
print(f"\nTotal geral: {df_stats['Total'].sum()} imagens")
print(f"Real: {df_stats['Real'].sum()} | Fake: {df_stats['Fake'].sum()}")

In [ ]:
# Dividir em train/val (80/20) e copiar para pasta unificada
random.shuffle(all_images)

split_idx = int(len(all_images) * 0.8)
train_images = all_images[:split_idx]
val_images = all_images[split_idx:]

print(f'Train: {len(train_images)} | Val: {len(val_images)}')

def copy_images(image_list, split_name):
    """Copia imagens para data/unified/{split}/{label}/ com nome unico."""
    counts = {'real': 0, 'fake': 0}
    for i, (src_path, label, source) in enumerate(image_list):
        ext = src_path.suffix
        dst_name = f'{source}_{i:06d}{ext}'
        dst_path = UNIFIED_DIR / split_name / label / dst_name
        try:
            shutil.copy2(src_path, dst_path)
            counts[label] += 1
        except Exception as e:
            pass  # skip arquivos com erro
        if (i + 1) % 10000 == 0:
            print(f'  {split_name}: {i+1}/{len(image_list)} copiadas...')
    return counts

print('Copiando train...')
train_counts = copy_images(train_images, 'train')
print(f'  Train: {train_counts}')

print('Copiando val...')
val_counts = copy_images(val_images, 'val')
print(f'  Val: {val_counts}')

print('\nDataset unificado pronto!')

In [ ]:
# Verificar contagem final
for split in ('train', 'val'):
    for cls in ('real', 'fake'):
        d = UNIFIED_DIR / split / cls
        n = len(list(d.glob('*')))
        print(f'{split}/{cls}: {n} imagens')

## 7. Configurar dataset e dataloader

Usa a pasta unificada com o MultiSourceDataset.

In [ ]:
import sys
sys.path.insert(0, '.')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import numpy as np

# Transforms
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class MultiSourceDataset(Dataset):
    """Dataset que carrega de data/unified/{split}/{real,fake}/."""

    LABELS = {'real': 0, 'fake': 1}

    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []

        for cls_name, cls_idx in self.LABELS.items():
            cls_dir = self.root_dir / cls_name
            if not cls_dir.exists():
                continue
            for img_path in cls_dir.iterdir():
                if img_path.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp', '.bmp'):
                    self.samples.append((str(img_path), cls_idx))

        print(f'  {root_dir}: {len(self.samples)} imagens '
              f'(real={sum(1 for _,l in self.samples if l==0)}, '
              f'fake={sum(1 for _,l in self.samples if l==1)})')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            # Imagem corrompida — retorna uma preta
            img = Image.new('RGB', (224, 224), (0, 0, 0))
        if self.transform:
            img = self.transform(img)
        return img, label


print('Criando datasets...')
train_ds = MultiSourceDataset('data/unified/train', transform=train_transforms)
val_ds = MultiSourceDataset('data/unified/val', transform=val_transforms)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)

print(f'\nBatches — Train: {len(train_loader)} | Val: {len(val_loader)}')

## 8. Carregar modelo

In [ ]:
from src.model import DeepShieldModel

device = torch.device('cuda')
model = DeepShieldModel(pretrained=True).to(device)

# Carregar pesos v1 se disponivel (transfer do modelo anterior)
v1_path = Path('models/best_model.pth')
if v1_path.exists():
    model.load_state_dict(torch.load(v1_path, map_location=device))
    print(f'Pesos v1 carregados de {v1_path}')
else:
    print('Usando pesos ImageNet (sem modelo v1 disponivel)')

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parametros: {total_params:,} total | {trainable:,} treinaveis')

## 9. Funcoes de treino e avaliacao

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import time
import json

def compute_class_weights(dataset):
    """Calcula pesos inversamente proporcionais a frequencia."""
    labels = [l for _, l in dataset.samples]
    counts = Counter(labels)
    total = len(labels)
    weights = torch.tensor([total / (2 * counts[i]) for i in range(2)],
                           dtype=torch.float32)
    print(f'Class weights: real={weights[0]:.3f}, fake={weights[1]:.3f}')
    return weights.to(device)


def train_one_epoch(model, loader, criterion, optimizer, epoch):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, desc=f'Epoch {epoch} [train]', leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    epoch_loss = running_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='binary')
    return epoch_loss, acc, f1


@torch.no_grad()
def evaluate(model, loader, criterion, epoch):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, desc=f'Epoch {epoch} [val]  ', leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='binary')
    prec = precision_score(all_labels, all_preds, average='binary', zero_division=0)
    rec = recall_score(all_labels, all_preds, average='binary', zero_division=0)
    return epoch_loss, acc, f1, prec, rec


print('Funcoes de treino prontas.')

## 10. Fase 1 — Backbone congelado (5 epochs)

Apenas o classificador e treinado. LR = 1e-3.

In [ ]:
# Congelar backbone
model.freeze_backbone()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Fase 1: {trainable:,} parametros treinaveis (apenas classificador)')

class_weights = compute_class_weights(train_ds)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=1e-3, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

PHASE1_EPOCHS = 5
history = {'phase1': [], 'phase2': []}
best_f1 = 0.0

print(f'\nIniciando Fase 1: {PHASE1_EPOCHS} epochs, lr=1e-3')
print('-' * 70)

for epoch in range(1, PHASE1_EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc, train_f1 = train_one_epoch(model, train_loader, criterion, optimizer, epoch)
    val_loss, val_acc, val_f1, val_prec, val_rec = evaluate(model, val_loader, criterion, epoch)
    scheduler.step()
    elapsed = time.time() - t0

    record = {
        'epoch': epoch, 'phase': 1,
        'train_loss': train_loss, 'train_acc': train_acc, 'train_f1': train_f1,
        'val_loss': val_loss, 'val_acc': val_acc, 'val_f1': val_f1,
        'val_prec': val_prec, 'val_rec': val_rec, 'time': elapsed,
    }
    history['phase1'].append(record)

    marker = ''
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), 'models/best_model_v2.pth')
        marker = ' <-- best'

    print(f'Epoch {epoch}/{PHASE1_EPOCHS} ({elapsed:.0f}s) | '
          f'Train loss={train_loss:.4f} acc={train_acc:.4f} | '
          f'Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}{marker}')

print(f'\nFase 1 concluida. Melhor F1: {best_f1:.4f}')

## 11. Fase 2 — Fine-tune ultimos 3 blocos (10 epochs)

Descongela os ultimos 3 blocos do EfficientNet. LR = 1e-4 com early stopping.

In [ ]:
# Descongelar ultimos 3 blocos
model.unfreeze_last_blocks(n=3)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Fase 2: {trainable:,} parametros treinaveis')

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=1e-4, weight_decay=0.01)
PHASE2_EPOCHS = 10
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE2_EPOCHS)

patience = 3
patience_counter = 0

print(f'\nIniciando Fase 2: {PHASE2_EPOCHS} epochs, lr=1e-4, patience={patience}')
print('-' * 70)

for epoch in range(1, PHASE2_EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc, train_f1 = train_one_epoch(model, train_loader, criterion, optimizer, epoch)
    val_loss, val_acc, val_f1, val_prec, val_rec = evaluate(model, val_loader, criterion, epoch)
    scheduler.step()
    elapsed = time.time() - t0

    record = {
        'epoch': epoch, 'phase': 2,
        'train_loss': train_loss, 'train_acc': train_acc, 'train_f1': train_f1,
        'val_loss': val_loss, 'val_acc': val_acc, 'val_f1': val_f1,
        'val_prec': val_prec, 'val_rec': val_rec, 'time': elapsed,
    }
    history['phase2'].append(record)

    marker = ''
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), 'models/best_model_v2.pth')
        patience_counter = 0
        marker = ' <-- best'
    else:
        patience_counter += 1

    print(f'Epoch {epoch}/{PHASE2_EPOCHS} ({elapsed:.0f}s) | '
          f'Train loss={train_loss:.4f} acc={train_acc:.4f} | '
          f'Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}{marker}')

    if patience_counter >= patience:
        print(f'\nEarly stopping! Sem melhora por {patience} epochs.')
        break

print(f'\nFase 2 concluida. Melhor F1 global: {best_f1:.4f}')

## 12. Salvar historico e comparar v1 vs v2

In [ ]:
# Salvar historico
history_path = Path('models/history_v2.json')
history_path.parent.mkdir(parents=True, exist_ok=True)
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f'Historico salvo em {history_path}')

# Metricas finais do melhor modelo v2
model.load_state_dict(torch.load('models/best_model_v2.pth', map_location=device))
val_loss, val_acc, val_f1, val_prec, val_rec = evaluate(model, val_loader, criterion, 0)

print(f'\n{"="*60}')
print(f'METRICAS FINAIS — DeepShield v2 (multi-gerador)')
print(f'{"="*60}')
print(f'Accuracy:  {val_acc:.4f}  ({val_acc:.2%})')
print(f'F1 Score:  {val_f1:.4f}')
print(f'Precision: {val_prec:.4f}  ({val_prec:.2%})')
print(f'Recall:    {val_rec:.4f}  ({val_rec:.2%})')

# Comparacao com v1
print(f'\n{"="*60}')
print(f'COMPARACAO v1 (StyleGAN) vs v2 (multi-gerador)')
print(f'{"="*60}')
print(f'{"Metrica":<12} {"v1 (StyleGAN)":>15} {"v2 (multi)":>15} {"Diff":>10}')
print(f'{"-"*52}')

v1_metrics = {'Accuracy': 0.9984, 'F1': 0.9984, 'Precision': 0.9985, 'Recall': 0.9983}
v2_metrics = {'Accuracy': val_acc, 'F1': val_f1, 'Precision': val_prec, 'Recall': val_rec}

for name in v1_metrics:
    v1_val = v1_metrics[name]
    v2_val = v2_metrics[name]
    diff = v2_val - v1_val
    sign = '+' if diff >= 0 else ''
    print(f'{name:<12} {v1_val:>14.4f} {v2_val:>14.4f} {sign}{diff:>9.4f}')

## 13. Graficos de treino

In [ ]:
import matplotlib.pyplot as plt

all_records = history['phase1'] + history['phase2']
epochs_range = list(range(1, len(all_records) + 1))
phase1_end = len(history['phase1'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('DeepShield v2 — Treino Multi-Gerador', fontsize=14, fontweight='bold')

# Loss
axes[0].plot(epochs_range, [r['train_loss'] for r in all_records], 'b-o', label='Train', markersize=4)
axes[0].plot(epochs_range, [r['val_loss'] for r in all_records], 'r-o', label='Val', markersize=4)
axes[0].axvline(x=phase1_end + 0.5, color='gray', linestyle='--', alpha=0.5, label='Fase 1|2')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, [r['train_acc'] for r in all_records], 'b-o', label='Train', markersize=4)
axes[1].plot(epochs_range, [r['val_acc'] for r in all_records], 'r-o', label='Val', markersize=4)
axes[1].axvline(x=phase1_end + 0.5, color='gray', linestyle='--', alpha=0.5, label='Fase 1|2')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# F1
axes[2].plot(epochs_range, [r['train_f1'] for r in all_records], 'b-o', label='Train', markersize=4)
axes[2].plot(epochs_range, [r['val_f1'] for r in all_records], 'r-o', label='Val', markersize=4)
axes[2].axvline(x=phase1_end + 0.5, color='gray', linestyle='--', alpha=0.5, label='Fase 1|2')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('F1 Score')
axes[2].set_title('F1 Score')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('models/training_v2_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafico salvo em models/training_v2_curves.png')

## 14. Download dos arquivos treinados

In [ ]:
from google.colab import files

print('Arquivos para download:')
print(f'  models/best_model_v2.pth')
print(f'  models/history_v2.json')
print(f'  models/training_v2_curves.png')

files.download('models/best_model_v2.pth')
files.download('models/history_v2.json')
files.download('models/training_v2_curves.png')

print('\nDeepShield v2 treinado com sucesso!')